# Debug: Claude Opus 4.7 Refusal

Submit a single benchmark query to `claude-opus-4-7` and inspect the full API response.

In [2]:
import os, sys
sys.path.insert(0, '../src')

import anthropic
from dotenv import load_dotenv
from utils import parse_csv, build_query

load_dotenv(override=True)

client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

In [3]:
# Build the first benchmark query
configs = parse_csv('../docs/virseq_benchmark.csv')
query = build_query(configs[0], use_gget_virus=False, return_integer_only=True)
print(f'Query (query_id={configs[0].query_id}, {configs[0].pathogen}):')
print(query)

Query (query_id=1, Crimean-Congo Hemorrhagic fever virus (CCHFV)):
Retrieve viral sequences from NCBI for TaxID 3052518 (Crimean-Congo Hemorrhagic fever virus (CCHFV)) that adhere to the following criteria: geographic location of sample collection: Africa, collected on or after 2020-01-01, collected on or before 2025-12-31, released on or after 2020-01-01, released on or before 2025-12-31, minimum sequence length: 11000 bp, maximum 10 ambiguous characters (N's) . Return the final count as a single integer on its own line.


In [4]:
from benchmark_claude import SYSTEM_PROMPT, EXECUTE_PYTHON_TOOL, WEB_SEARCH_TOOL

print('System prompt:')
print(SYSTEM_PROMPT)

System prompt:
You are a bioinformatics agent.
You have access to:
- web_search: search the internet for up-to-date information.
- execute_python: run Python LOCALLY (WITH internet) to call APIs, install packages, and compute results. Note: outbound network access is restricted to an allowlist of domains.

Workflow:
1) If you need to choose the best API/library, use web_search first.
2) Then use execute_python to implement the call and compute the final answer.

Output rules:
- When you have the final count, respond with exactly one integer on its own line.
- After you output the final integer, also output a JSON block with keys: methods (eg APIs used) and reasoning (max 3 bullets). Keep it short.



In [ ]:
# Models: https://platform.claude.com/docs/en/about-claude/models/overview
# MODEL = 'claude-opus-4-5-20251101'
# MODEL = 'claude-opus-4-6'
MODEL = 'claude-opus-4-7'
# MODEL = 'claude-sonnet-4-20250514'
# MODEL = 'claude-sonnet-4-5-20250929'
# MODEL = 'claude-sonnet-4-6'

response = client.messages.create(
    model=MODEL,
    max_tokens=4096,
    system=SYSTEM_PROMPT,
    tools=[EXECUTE_PYTHON_TOOL, WEB_SEARCH_TOOL],
    messages=[{'role': 'user', 'content': query}],
)

print(f'stop_reason: {response.stop_reason}')
print(f'model: {response.model}')
print(f'usage: {response.usage}')
print(f'content blocks: {len(response.content)}')
print()
for i, block in enumerate(response.content):
    print(f'--- Block {i} ---')
    print(f'type: {block.type}')
    print(block)
    print()

stop_reason: refusal
model: claude-opus-4-5-20251101
usage: Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=2635, output_tokens=1, server_tool_use=None, service_tier='standard')
content blocks: 0



In [6]:
# # Try without tools to see if refusal is query-related or tool-related
# response_no_tools = client.messages.create(
#     model=MODEL,
#     max_tokens=4096,
#     system=SYSTEM_PROMPT,
#     messages=[{'role': 'user', 'content': query}],
# )

# print(f'stop_reason: {response_no_tools.stop_reason}')
# print(f'usage: {response_no_tools.usage}')
# print()
# for i, block in enumerate(response_no_tools.content):
#     print(f'--- Block {i} ---')
#     print(f'type: {block.type}')
#     print(block)
#     print()

In [7]:
# # Try with higher max_tokens in case thinking tokens are the issue
# response_high = client.messages.create(
#     model=MODEL,
#     max_tokens=16384,
#     system=SYSTEM_PROMPT,
#     tools=[EXECUTE_PYTHON_TOOL, WEB_SEARCH_TOOL],
#     messages=[{'role': 'user', 'content': query}],
# )

# print(f'stop_reason: {response_high.stop_reason}')
# print(f'usage: {response_high.usage}')
# print()
# for i, block in enumerate(response_high.content):
#     print(f'--- Block {i} ---')
#     print(f'type: {block.type}')
#     print(block)
#     print()